# 09 - Community Detection

Everything up to this point has treated the Israeli public-transport network as one object and asked how fragile it is. This notebook asks a different question: **does the network split, on its own, into internally well-connected blocks?** We run the **Louvain** modularity-maximisation algorithm on the weighted undirected trip-adjacency graph, measure how good the resulting partition is (**modularity**), describe the communities it finds, and then identify the **inter-community bridge stations** - the stops whose neighbours lie in more than one community and which therefore carry the traffic that has to cross a community boundary.

**Research question addressed here:** is the network a set of loosely coupled regional/metropolitan clusters rather than one homogeneous mesh, and which stations hold those clusters together?

### A discrepancy this notebook is written to settle

Two earlier write-ups of this project disagree: the Hebrew final report states that Louvain found **91** communities, while the presentation pipeline reports **73**. Neither number is wrong in the sense of being a mistake - Louvain is a *stochastic greedy heuristic*, so the number of communities it returns is not a fixed property of the graph. It depends on the random seed, on the resolution parameter, on the implementation, and on exactly which graph was fed in. This notebook therefore:

1. exposes `LOUVAIN_SEED` and `LOUVAIN_RESOLUTION` as explicit constants at the top,
2. reports whatever **this** run produces, with its modularity, instead of quoting a remembered figure,
3. runs a seed sweep and a resolution sweep to show empirically how much the count moves while the partition itself stays essentially the same,
4. explains, in section 8, which of these mechanisms can produce a gap of the size seen between the two earlier reports.

No community count and no community id is written into the code or the prose anywhere below; every number and every label is computed from the data at run time.

## Inputs

* `outputs/nb/02_graph_construction/nodes.csv` - one row per active stop with `stop_id, stop_name, lat, lon, region, metro`.
* `outputs/nb/02_graph_construction/edges.csv` - one row per **directed** segment `from_stop, to_stop, trip_frequency`.

Both are produced by notebook `02_graph_construction`. This notebook does **not** read the raw GTFS feed, does **not** need the 816 MB `stop_times.txt`, and does **not** import anything from `src/` or `public_transport_network_research/` - all logic is inline.

## Outputs

Everything is written under `outputs/nb/09_community_detection/`:

* `community_detection_summary.json` - headline statistics (seed, resolution, backend, community count, modularity, boundary statistics) in one dictionary.
* `tables/community_assignments.csv` - one row per station with its Louvain community (and its label-propagation community, for comparison).
* `tables/community_summary.csv` - one row per community: size, internal/external edges and weight, centroid, dominant region and metropolitan area, hub station, derived label.
* `tables/inter_community_bridges.csv` - every station whose neighbourhood spans more than one community, ranked by how many communities it touches.
* `tables/louvain_stability.csv` - the seed sweep and resolution sweep results (communities, modularity, agreement with the headline run).
* `tables/community_detection_summary.csv` - the headline statistics as a one-row table.
* `figures/community_map_louvain.png` - every station drawn at its true coordinates, coloured by community.
* `figures/community_size_distribution.png` - the largest communities as a bar chart plus the full rank-size distribution.
* `figures/inter_community_bridges_map.png` - the boundary-spanning stations on the map.
* `figures/louvain_stability.png` - community count and modularity across seeds and resolutions.

Nothing outside `outputs/nb/09_community_detection/` is touched; in particular the pre-existing `outputs/tables`, `outputs/figures` and `outputs/rail` folders, which hold the results cited in the written report, are never written to.


## 1. Environment bootstrap

The cell below makes the notebook runnable both on a local checkout and on Google Colab. It defines `_ensure(...)`, which pip-installs only the packages that are genuinely missing (so re-running the notebook is cheap), and `find_repo_root()`, which walks up from the current directory looking for the GTFS folder and, failing that, clones the repository into `/content`. It then sets `REPO`, `DATA` and `OUT` and creates the notebook output root. Every later cell relies on these three paths, so this must run first. This is the same bootstrap used by every other notebook in the series, kept identical on purpose so that any of them can be run standalone.


In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)


## 2. Libraries, stage folders and tunable constants

We install and import the scientific stack and then fix the folder layout for this stage. Following the project convention, each notebook owns exactly one output folder: this one writes into `outputs/nb/09_community_detection/` (with `tables/` and `figures/` sub-folders) and reads the previous stage from `outputs/nb/02_graph_construction/`.

Two Louvain implementations exist in the Python ecosystem and they do **not** always return the same partition: the standalone `python-louvain` package (imported as `community`, the one the original project script used) and the implementation bundled with `networkx` since version 2.8. We prefer the former for continuity with the earlier pipeline and fall back to the latter if the package cannot be installed - the choice is recorded in `LOUVAIN_BACKEND` and written into the summary JSON, because "which implementation" is itself one of the explanations for the 91-vs-73 discrepancy.

The constants are gathered here so a grader can change the experiment in one place:

* `LOUVAIN_SEED` - the RNG seed of the headline run. Louvain visits nodes in a randomised order and breaks ties randomly, so this genuinely changes the answer. Fixing it makes this notebook reproducible.
* `LOUVAIN_RESOLUTION` - the parameter `gamma` in the modularity objective. `1.0` is the classical Newman-Girvan definition; values above 1 penalise large communities and therefore return more, smaller ones; values below 1 return fewer, larger ones.
* `SEED_SWEEP` / `RESOLUTION_SWEEP` - the two sensitivity experiments of section 8.
* `RUN_LABEL_PROPAGATION` - also run a second, completely different community algorithm as a cross-check.
* the remaining knobs control figure resolution and how many rows the top-N tables and charts show.

Louvain on a roughly 30k-node / 52k-edge graph takes a few seconds per run, so the sweeps below cost about a dozen runs and still finish in well under a minute.


In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn', 'python-louvain', 'scikit-learn')

import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

# Prefer the standalone python-louvain package (what the original script used);
# fall back to the implementation shipped with networkx.
try:
    import community as community_louvain
    LOUVAIN_BACKEND = 'python-louvain'
except ImportError:
    community_louvain = None
    LOUVAIN_BACKEND = 'networkx'

PREV = OUT / '02_graph_construction'      # read-only: artifacts of notebook 02
STAGE = OUT / '09_community_detection'    # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
LOUVAIN_SEED = 42                              # RNG seed of the headline run
LOUVAIN_RESOLUTION = 1.0                       # gamma in the modularity objective
SEED_SWEEP = [0, 1, 7, 42, 123, 2024]          # seeds tried in the stability experiment
RESOLUTION_SWEEP = [0.5, 0.8, 1.0, 1.2, 2.0]   # resolutions tried at the fixed seed
RUN_LABEL_PROPAGATION = True                   # second algorithm, used only as a cross-check
WEIGHT_ATTR = 'weight'                         # edge attribute Louvain optimises over
FIG_DPI = 150                                  # figure resolution; drop to 90 for smaller files
TOP_N = 20                                     # rows shown in every top-N table / bar chart
MAP_ALPHA = 0.6                                # point transparency on the ~30k-point maps
MAP_POINT_SIZE = 3                             # marker size on the community map
ANNOTATE_TOP = 8                               # communities labelled in place on the map

print('previous stage :', PREV)
print('this stage     :', STAGE)
print('Louvain backend:', LOUVAIN_BACKEND)
print(f'seed = {LOUVAIN_SEED}, resolution = {LOUVAIN_RESOLUTION}')


## 3. Hebrew label rendering

Stop names in the Israeli GTFS feed are Hebrew, and several figures below print them (the community bar chart labels each community with its busiest station). Matplotlib does not implement the Unicode bidirectional algorithm, so right-to-left text comes out reversed and unreadable. The cell below monkey-patches `matplotlib.text.Text.set_text` once so that any string containing Hebrew characters is converted to display order via `python-bidi` before it is drawn, and selects a font that actually has Hebrew glyphs (Arial on Windows, DejaVu Sans everywhere else). It is idempotent - re-running it will not stack patches. All other text in the notebook is English, per the submission requirement.


In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()


## 4. Loading the graph produced by notebook 02

This stage depends on notebook `02_graph_construction`. Rather than un-pickling a `networkx` object (pickles are version-fragile and unreadable to a grader), we reload the two plain CSV tables that stage 02 exports and rebuild the graph from them - the CSVs contain exactly the same information. This is a deliberate change from the original project script, which loaded `graph_undirected.pkl`. `find_artifact` searches the whole previous-stage folder so it works whether stage 02 put its tables at the folder root or inside `tables/`, and raises an explicit, actionable error if the artifacts are missing.


In [ ]:
# --- Locate the artifacts written by notebook 02 --------------------------
def find_artifact(stage_dir, filename):
    """Return the path of `filename` under a stage folder, or None if absent."""
    if not stage_dir.is_dir():
        return None
    direct = stage_dir / filename
    if direct.exists():
        return direct
    matches = sorted(stage_dir.rglob(filename))
    return matches[0] if matches else None

nodes_path = find_artifact(PREV, 'nodes.csv')
edges_path = find_artifact(PREV, 'edges.csv')
missing = [name for name, p in [('nodes.csv', nodes_path), ('edges.csv', edges_path)] if p is None]
if missing:
    raise FileNotFoundError(
        f"{', '.join(missing)} not found under {PREV} - "
        'run notebook 02_graph_construction first; it writes nodes.csv and edges.csv.'
    )

nodes_df = pd.read_csv(nodes_path, dtype={'stop_id': str}, encoding='utf-8-sig')
edges_df = pd.read_csv(edges_path, dtype={'from_stop': str, 'to_stop': str}, encoding='utf-8-sig')
print(f'nodes: {len(nodes_df):,} rows  <-  {nodes_path}')
print(f'edges: {len(edges_df):,} rows  <-  {edges_path}')
nodes_df.head()


## 5. Rebuilding the weighted undirected graph

The project's model is a **trip-adjacency graph**: a node is a stop that appears in at least one trip, and a directed edge `u -> v` exists when some trip visits `v` immediately after `u`; the edge weight is the number of trips using that segment. `edges.csv` stores this directed graph. Community detection is defined for undirected graphs, so we take the undirected projection: one edge per unordered pair, with the weights of the two travel directions **summed**. That summed weight is what Louvain optimises over, which means a segment served by a hundred buses an hour pulls its two endpoints into the same community far more strongly than a segment served twice a day - the partition is about service intensity, not just topology.

Station attributes (name, coordinates, region, metropolitan area) are attached from `nodes.csv`. The helpers `_num` and `_txt` coerce blanks and `NaN` safely, so a missing coordinate becomes `None` rather than a silent `NaN` that would later be plotted at a nonsense position.


In [ ]:
# --- Rebuild the weighted undirected graph ---------------------------------
def _num(value):
    """Coerce to float; return None for blanks, NaN or non-numeric input."""
    if value is None:
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(f) else f


def _txt(value):
    """Coerce to a plain string; NaN and None become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ''
    return str(value)


def build_undirected_graph(nodes_df, edges_df):
    """Undirected projection of the directed trip graph; opposite-direction weights summed."""
    attr = {}
    for rec in nodes_df.to_dict('records'):
        attr[str(rec.get('stop_id'))] = {
            'stop_name': _txt(rec.get('stop_name')),
            'lat': _num(rec.get('lat')),
            'lon': _num(rec.get('lon')),
            'region': _txt(rec.get('region')),
            'metro': _txt(rec.get('metro')),
        }

    weight_col = next((c for c in ('trip_frequency', 'weight', 'count')
                       if c in edges_df.columns), None)

    G = nx.Graph()
    for rec in edges_df.to_dict('records'):
        u, v = str(rec['from_stop']), str(rec['to_stop'])
        if u == v:
            continue
        w = int(rec[weight_col]) if weight_col else 1
        if G.has_edge(u, v):
            G[u][v][WEIGHT_ATTR] += w
        else:
            G.add_edge(u, v, **{WEIGHT_ATTR: w})

    default = {'stop_name': '', 'lat': None, 'lon': None, 'region': '', 'metro': ''}
    for n in G.nodes():
        G.nodes[n].update(attr.get(n, default))
    return G


G = build_undirected_graph(nodes_df, edges_df)
total_weight = sum(d[WEIGHT_ATTR] for _, _, d in G.edges(data=True))
unnamed = sum(1 for n in G.nodes() if not G.nodes[n]['stop_name'])
print(f'undirected G : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'total edge weight (trips over all segments): {total_weight:,}')
print(f'nodes with no name attribute: {unnamed:,}')


## 6. Restricting the analysis to the largest connected component

The original project script ran Louvain on `G.subgraph(max(nx.connected_components(G), key=len))`, i.e. on the **largest connected component only**, and we keep that choice. The reason is that isolated components are trivially their own communities: an island of six stops that touches nothing else *must* form a separate community under any partition, and counting each of them inflates the community count without saying anything about the structure of the network. Notebook 03 showed that the largest component holds nearly all stations, so almost nothing is lost.

This is also the first concrete candidate explanation for the 91-vs-73 disagreement: a run on the **full** graph automatically reports `(number of components - 1)` more communities than a run on the largest component alone, before any randomness is involved. The cell prints the exact size of that offset for this graph, so the effect can be quantified rather than guessed at. Stations outside the largest component are still exported in `community_assignments.csv`, with community id `-1` meaning "not analysed".


In [ ]:
# --- Largest connected component ------------------------------------------
components = sorted(nx.connected_components(G), key=len, reverse=True)
Gc = G.subgraph(components[0]).copy()

outside = G.number_of_nodes() - Gc.number_of_nodes()
print(f'connected components in G          : {len(components):,}')
print(f'largest component (analysed)       : {Gc.number_of_nodes():,} nodes, '
      f'{Gc.number_of_edges():,} edges '
      f'({Gc.number_of_nodes() / G.number_of_nodes():.2%} of all stations)')
print(f'stations outside it (community -1) : {outside:,}')
print(f'running on the full graph instead would add {len(components) - 1} trivial communities')


## 7. Louvain: what it optimises, and the headline run

Louvain is a greedy heuristic for maximising **modularity**

$$Q \;=\; \frac{1}{2m}\sum_{i,j}\left[A_{ij} - \gamma\,\frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$$

where $A_{ij}$ is the summed trip weight between stations $i$ and $j$, $k_i$ is the weighted degree of $i$, $m$ is the total edge weight, $\gamma$ is the resolution and $\delta(c_i,c_j)$ is 1 when the two stations are in the same community. In words: $Q$ compares how much weight actually falls **inside** communities against how much would fall inside them if the same stations kept their weighted degrees but were wired at random. $Q$ near 0 means "no better than random"; values roughly in the 0.3-0.7 range are the usual sign of genuine community structure; the theoretical maximum is 1.

The algorithm alternates two phases until $Q$ stops improving: (1) **local moving** - visit the nodes in some order and move each one into the neighbouring community that gives the largest modularity gain; (2) **aggregation** - collapse each community into a single super-node and repeat on the smaller graph. Both phases are cheap, which is why Louvain scales to graphs of this size in seconds.

**The stochastic part is phase 1**: the order in which nodes are visited, and how ties are broken, come from the RNG. Different orders lead to different local optima of a non-convex objective, and those optima can have nearly identical modularity while differing in how many communities they contain - typically because a handful of borderline clusters merge or split. That is precisely why we pin `LOUVAIN_SEED`.

Three helpers do the work below. `louvain_partition` wraps whichever backend is available behind one signature that always takes an explicit seed and resolution. `partition_modularity` scores any partition with `networkx`, so the modularity numbers stay comparable across backends and across the sweeps. `relabel_by_size` renumbers the communities so that id 0 is the largest, id 1 the second largest and so on, with a deterministic tie-break - the raw ids Louvain emits are arbitrary and would change between runs, which would make the exported tables impossible to compare.


In [ ]:
# --- Louvain helpers -------------------------------------------------------
def louvain_partition(graph, seed, resolution, weight=WEIGHT_ATTR):
    """One Louvain run -> {node: community_id}. Seed and resolution are always explicit."""
    if community_louvain is not None:
        return dict(community_louvain.best_partition(
            graph, weight=weight, resolution=resolution, random_state=seed))
    communities = nx.community.louvain_communities(
        graph, weight=weight, resolution=resolution, seed=seed)
    return {node: cid for cid, members in enumerate(communities) for node in members}


def groups_from_partition(partition):
    """{node: cid} -> {cid: set(nodes)}."""
    groups = defaultdict(set)
    for node, cid in partition.items():
        groups[cid].add(node)
    return groups


def partition_modularity(graph, partition, weight=WEIGHT_ATTR, resolution=1.0):
    """Newman-Girvan modularity of a partition, scored with networkx for comparability."""
    groups = groups_from_partition(partition)
    return float(nx.community.modularity(
        graph, list(groups.values()), weight=weight, resolution=resolution))


def relabel_by_size(partition):
    """Renumber communities so id 0 is the largest; ties broken deterministically."""
    groups = groups_from_partition(partition)
    order = sorted(groups.items(), key=lambda kv: (-len(kv[1]), min(str(n) for n in kv[1])))
    mapping = {old: new for new, (old, _) in enumerate(order)}
    return {node: mapping[cid] for node, cid in partition.items()}


# --- Headline run ----------------------------------------------------------
partition = relabel_by_size(louvain_partition(Gc, LOUVAIN_SEED, LOUVAIN_RESOLUTION))
modularity = partition_modularity(Gc, partition, resolution=LOUVAIN_RESOLUTION)

community_sizes = pd.Series(Counter(partition.values())).sort_values(ascending=False)
num_communities = int(community_sizes.size)

print(f'backend                : {LOUVAIN_BACKEND}')
print(f'seed / resolution      : {LOUVAIN_SEED} / {LOUVAIN_RESOLUTION}')
print(f'communities found      : {num_communities:,}')
print(f'modularity Q           : {modularity:.4f}')
print(f'largest community      : {int(community_sizes.iloc[0]):,} stations '
      f'({community_sizes.iloc[0] / Gc.number_of_nodes():.2%} of the component)')
print(f'median community size  : {community_sizes.median():.0f}')
print(f'communities with fewer than 10 stations: {int((community_sizes < 10).sum()):,}')


## 8. How stable is that number? Seed sweep, resolution sweep, and the 91-vs-73 gap

The count printed above is one draw from a distribution, not a constant. This cell re-runs Louvain across `SEED_SWEEP` (resolution held at `LOUVAIN_RESOLUTION`) and across `RESOLUTION_SWEEP` (seed held at `LOUVAIN_SEED`) and records, for each run, the number of communities, the modularity, the size of the largest community, and the **adjusted Rand index (ARI)** against the headline partition.

ARI is the key column. It measures how often two partitions agree about whether a *pair* of stations belongs together, corrected for the agreement expected by chance: 1.0 means identical partitions, 0.0 means no better than random. If the seed sweep produces a spread of community counts but ARI values close to 1, the honest conclusion is that **the partition is stable and only the count is noisy** - a few small clusters merge or split at the margins while the large regional blocks stay put. If ARI were low, the whole partition would be unstable and no community-level claim could be trusted.

This is the empirical answer to the report-versus-presentation discrepancy. The mechanisms that can move the community count, none of which are errors, are:

1. **Random seed.** Different node visit orders give different local optima of a non-convex objective. Quantified directly by the seed sweep below.
2. **Resolution `gamma`.** The count grows with `gamma`: raising it splits communities, lowering it merges them. A pipeline that left the resolution at one library's default while another used a different default will report a different number. Quantified by the resolution sweep.
3. **Which graph was partitioned.** Full graph versus largest component (section 6 printed that exact offset); weighted versus unweighted edges; opposite directions collapsed by summing versus by taking a maximum. All change the objective, and therefore the answer.
4. **Which implementation.** `python-louvain` and the `networkx` implementation differ in tie-breaking, in their convergence threshold, and in how they refine the aggregated graph; igraph's Leiden refinement differs more still. Same algorithm name, different local optimum. The backend actually used here is printed in section 2 and stored in the summary JSON.
5. **Which snapshot of the GTFS feed.** Rebuilding the graph from a later feed changes the node and edge sets slightly, which perturbs the partition.

A difference of a couple of dozen communities is entirely consistent with mechanisms 1-4 acting together, especially since most of the communities near the split/merge margin are small. What deserves to be quoted in a report is not the count but the **modularity**, the **size distribution** and the **spatial structure** - all three of which are far more reproducible, as the table below shows.


In [ ]:
# --- Seed sweep and resolution sweep ---------------------------------------
from sklearn.metrics import adjusted_rand_score

node_order = list(Gc.nodes())
base_labels = [partition[n] for n in node_order]


def stability_row(experiment, seed, resolution):
    """Run Louvain once and describe the result relative to the headline partition."""
    p = louvain_partition(Gc, seed, resolution)
    sizes = Counter(p.values())
    return {
        'experiment': experiment,
        'seed': seed,
        'resolution': resolution,
        'num_communities': len(sizes),
        'modularity': round(partition_modularity(Gc, p, resolution=resolution), 4),
        'largest_community': max(sizes.values()),
        'communities_under_10': sum(1 for s in sizes.values() if s < 10),
        'ari_vs_headline': round(
            float(adjusted_rand_score(base_labels, [p[n] for n in node_order])), 4),
    }


stability = pd.DataFrame(
    [stability_row('seed sweep', s, LOUVAIN_RESOLUTION) for s in SEED_SWEEP]
    + [stability_row('resolution sweep', LOUVAIN_SEED, r) for r in RESOLUTION_SWEEP]
)
stability.to_csv(TABLES / 'louvain_stability.csv', index=False, encoding='utf-8-sig')

seed_runs = stability[stability['experiment'] == 'seed sweep']
spread = int(seed_runs['num_communities'].max() - seed_runs['num_communities'].min())
print('across seeds, at the fixed resolution:')
print(f"  communities : {int(seed_runs['num_communities'].min())} to "
      f"{int(seed_runs['num_communities'].max())}  (spread {spread})")
print(f"  modularity  : {seed_runs['modularity'].min():.4f} to {seed_runs['modularity'].max():.4f}")
print(f"  ARI vs headline : {seed_runs['ari_vs_headline'].min():.3f} to "
      f"{seed_runs['ari_vs_headline'].max():.3f}")
stability


### 8b. The same experiment as a figure

The left panel plots the community count against the seed with the modularity of each run printed on top of its bar, so the two can be read together: if the bars vary in height while the modularity values barely move, the objective function is flat across those solutions and the count is close to arbitrary. The right panel plots community count and modularity against the resolution `gamma` on twin axes, showing that deliberately choosing `gamma` moves the count far more than randomness does. The dashed reference lines mark the headline run.


In [ ]:
# --- figures/louvain_stability.png -----------------------------------------
seed_df = stability[stability['experiment'] == 'seed sweep'].sort_values('seed')
res_df = stability[stability['experiment'] == 'resolution sweep'].sort_values('resolution')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

bars = axes[0].bar(seed_df['seed'].astype(str), seed_df['num_communities'], color='#2563eb')
for bar, q in zip(bars, seed_df['modularity']):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'Q={q:.4f}', ha='center', va='bottom', fontsize=9)
axes[0].axhline(num_communities, color='#111827', linestyle='--', linewidth=1,
                label='headline run')
axes[0].set_xlabel('random seed')
axes[0].set_ylabel('number of communities')
axes[0].set_title(f'Community count across seeds (resolution = {LOUVAIN_RESOLUTION})')
axes[0].margins(y=0.15)
axes[0].legend()

axes[1].plot(res_df['resolution'], res_df['num_communities'], marker='o', color='#7c3aed')
axes[1].set_xlabel('resolution gamma')
axes[1].set_ylabel('number of communities', color='#7c3aed')
axes[1].tick_params(axis='y', labelcolor='#7c3aed')
axes[1].axvline(LOUVAIN_RESOLUTION, color='#111827', linestyle='--', linewidth=1)
twin = axes[1].twinx()
twin.plot(res_df['resolution'], res_df['modularity'], marker='s', color='#dc2626')
twin.set_ylabel('modularity Q', color='#dc2626')
twin.tick_params(axis='y', labelcolor='#dc2626')
twin.grid(False)
axes[1].set_title(f'Community count and modularity vs resolution (seed = {LOUVAIN_SEED})')

plt.tight_layout()
plt.savefig(FIGURES / 'louvain_stability.png', dpi=FIG_DPI)
plt.show()


## 9. A second opinion: label propagation

The original script also ran **label propagation**, and we keep it as a cross-check. It is a completely different idea: every node starts in its own community and then repeatedly adopts the label held by most of its neighbours, until no node wants to change. It optimises nothing explicitly - there is no modularity term - so agreement between the two methods is meaningful evidence that the structure is real rather than an artefact of the modularity objective.

Two caveats, stated because they matter for interpreting the comparison. First, `networkx`'s `label_propagation_communities` ignores edge weights, so it sees only topology while Louvain sees service intensity; the two are answering slightly different questions. Second, label propagation is known for producing a very different *granularity* - often one very large community plus a long tail of small ones - which is another reminder that "number of communities" is a property of the method at least as much as of the network. We score its partition with the same modularity function so the two are on a comparable scale, and we reuse the same size-ordered relabelling.


In [ ]:
# --- Label propagation (unweighted, cross-check only) ----------------------
if RUN_LABEL_PROPAGATION:
    lp_communities = list(nx.community.label_propagation_communities(Gc))
    partition_lp = relabel_by_size(
        {node: cid for cid, members in enumerate(lp_communities) for node in members})
    modularity_lp = partition_modularity(Gc, partition_lp, resolution=LOUVAIN_RESOLUTION)
    lp_sizes = pd.Series(Counter(partition_lp.values())).sort_values(ascending=False)
    ari_lp = float(adjusted_rand_score(base_labels, [partition_lp[n] for n in node_order]))
    print(f'label propagation communities : {int(lp_sizes.size):,}')
    print(f'label propagation modularity  : {modularity_lp:.4f}   (Louvain: {modularity:.4f})')
    print(f'largest community             : {int(lp_sizes.iloc[0]):,} stations')
    print(f'agreement with Louvain (ARI)  : {ari_lp:.3f}')
else:
    partition_lp, modularity_lp, ari_lp = {}, None, None
    print('label propagation skipped (RUN_LABEL_PROPAGATION = False)')


## 10. Exporting the per-station community assignment

The first of the required tables. One row per station in the **whole** graph - not just the largest component - carrying its identity, coordinates, region, metropolitan area, degree and its community under each algorithm. Stations outside the largest component (and, if label propagation was skipped, every station in the label-propagation column) get `-1`, a sentinel meaning "not assigned by this run" that is never a real community id. Keeping the excluded stations in the file rather than dropping them means downstream notebooks can join on the full station list without silently losing rows.


In [ ]:
# --- tables/community_assignments.csv --------------------------------------
assign_df = pd.DataFrame([{
    'stop_id': n,
    'stop_name': G.nodes[n]['stop_name'],
    'lat': G.nodes[n]['lat'],
    'lon': G.nodes[n]['lon'],
    'region': G.nodes[n]['region'],
    'metro': G.nodes[n]['metro'],
    'degree': G.degree(n),
    'community_louvain': partition.get(n, -1),
    'community_lp': partition_lp.get(n, -1),
} for n in G.nodes()])
assign_df = assign_df.sort_values(['community_louvain', 'degree'],
                                  ascending=[True, False]).reset_index(drop=True)
assign_df.to_csv(TABLES / 'community_assignments.csv', index=False, encoding='utf-8-sig')

print(f"saved {TABLES / 'community_assignments.csv'}  ({len(assign_df):,} rows)")
print(f"unassigned stations (community -1): {int((assign_df['community_louvain'] == -1).sum()):,}")
assign_df.head(10)


## 11. Describing each community - and labelling it from the data

The second required table. For every community we compute, in a single pass over the edges (so the cost stays linear rather than quadratic in the number of communities, which is what the original script's per-community `subgraph` call would have cost):

* **size** - number of stations, and its share of the analysed component.
* **internal edges / internal weight** - segments and trips that stay inside the community.
* **external edges / external weight** - segments and trips that leave it. The ratio `external_weight / (internal_weight + external_weight)` is a conductance-style measure of how leaky the community is: near 0 means a self-contained cluster, near 1 means a group that mostly talks to the outside.
* **boundary stations** - how many of its stations have at least one neighbour in another community.
* **centroid** - mean latitude and longitude over its stations with usable coordinates, which is what places the label on the map.
* **dominant region / dominant metro** and their shares - the modal value of the `region` and `metro` attributes that notebook 01 assigned geographically.
* **hub station** - the member with the highest weighted degree, i.e. the busiest stop in the community.

The **label** is derived, never hard-coded: it is the community's own modal metropolitan area joined with its modal region (de-duplicated when the two coincide), and it is only as good as the geography assigned upstream in notebook 01 - a crude latitude/longitude rule with four metropolitan discs, not an administrative gazetteer. The `dominant_metro_share` column is exported precisely so a reader can see when a label is weak: a community whose modal metro covers only half its stations is a genuine mix that a single-word label flattens, and should be read together with the hub station name.


In [ ]:
# --- tables/community_summary.csv ------------------------------------------
groups = groups_from_partition(partition)

internal_edges = Counter(); internal_weight = Counter()
external_edges = Counter(); external_weight = Counter()
boundary_nodes = defaultdict(set)

for u, v, data in Gc.edges(data=True):
    cu, cv = partition[u], partition[v]
    w = data.get(WEIGHT_ATTR, 1)
    if cu == cv:
        internal_edges[cu] += 1
        internal_weight[cu] += w
    else:
        for c in (cu, cv):
            external_edges[c] += 1
            external_weight[c] += w
        boundary_nodes[cu].add(u)
        boundary_nodes[cv].add(v)

weighted_degree = dict(Gc.degree(weight=WEIGHT_ATTR))


def _mode(values):
    """Most common non-empty value and its share; ('', 0.0) when nothing is present."""
    counts = Counter(v for v in values if v)
    if not counts:
        return '', 0.0
    value, count = counts.most_common(1)[0]
    return value, round(count / len(values), 4)


def _label(metro, region):
    """Derived, data-driven community label: modal metro joined with modal region."""
    parts = [p for p in (metro, region) if p]
    return ' / '.join(dict.fromkeys(parts)) if parts else 'unlabelled'


rows = []
for cid, members in sorted(groups.items(), key=lambda kv: kv[0]):
    members = sorted(members)
    lats = [G.nodes[n]['lat'] for n in members if G.nodes[n]['lat'] is not None]
    lons = [G.nodes[n]['lon'] for n in members if G.nodes[n]['lon'] is not None]
    metro, metro_share = _mode([G.nodes[n]['metro'] for n in members])
    region, region_share = _mode([G.nodes[n]['region'] for n in members])
    hub = max(members, key=lambda n: (weighted_degree.get(n, 0), Gc.degree(n)))
    iw, ew = int(internal_weight[cid]), int(external_weight[cid])
    rows.append({
        'community_id': cid,
        'label': _label(metro, region),
        'size': len(members),
        'share_of_component': round(len(members) / Gc.number_of_nodes(), 5),
        'internal_edges': int(internal_edges[cid]),
        'external_edges': int(external_edges[cid]),
        'internal_weight': iw,
        'external_weight': ew,
        'external_weight_ratio': round(ew / (iw + ew), 4) if (iw + ew) else None,
        'boundary_stations': len(boundary_nodes[cid]),
        'boundary_share': round(len(boundary_nodes[cid]) / len(members), 4),
        'centroid_lat': round(float(np.mean(lats)), 4) if lats else None,
        'centroid_lon': round(float(np.mean(lons)), 4) if lons else None,
        'dominant_region': region,
        'dominant_region_share': region_share,
        'dominant_metro': metro,
        'dominant_metro_share': metro_share,
        'hub_stop_id': hub,
        'hub_stop_name': G.nodes[hub]['stop_name'],
    })

summary_df = pd.DataFrame(rows).sort_values('size', ascending=False).reset_index(drop=True)
summary_df.to_csv(TABLES / 'community_summary.csv', index=False, encoding='utf-8-sig')

print(f"saved {TABLES / 'community_summary.csv'}  ({len(summary_df):,} communities)")
print('communities per dominant metropolitan area (labels are derived, not hard-coded):')
print(summary_df['dominant_metro'].value_counts().to_string())
summary_df.head(TOP_N)[['community_id', 'label', 'size', 'internal_edges', 'external_edges',
                        'external_weight_ratio', 'dominant_metro_share', 'hub_stop_name']]


## 12. Inter-community bridge stations

The third required table, and the part that connects this notebook back to the project's resilience question. A station is an **inter-community bridge** when at least one of its neighbours sits in a different community: it is a point where a journey has to leave one cluster and enter another. Note that this is a *different* notion from the graph-theoretic bridges of notebook 03 - those were edges whose removal disconnects the graph outright, while these are stations that carry cross-cluster traffic and whose loss would force long detours even when the network technically stays connected. A station can be one, the other, both, or neither, and the two tables are worth joining.

For each such station we record how many **distinct** communities its neighbourhood touches (the original script's headline metric), how many of its incident edges cross a boundary, its degree and weighted degree, and what share of its own traffic is cross-community. Ranking by distinct communities first and by crossing weight second surfaces the stations that are both structurally central and heavily used - the ones a resilience study should stress-test first. A station touching many communities is an interchange between regional systems; a station touching one other community but carrying enormous crossing weight is a single high-volume corridor.

The cell also aggregates the two network-level quantities that say how tightly the clusters are coupled: the share of edges and the share of total trip weight that cross a community boundary.


In [ ]:
# --- tables/inter_community_bridges.csv ------------------------------------
label_of = dict(zip(summary_df['community_id'], summary_df['label']))

bridge_rows = []
for node in Gc.nodes():
    own = partition[node]
    other_comms = set()
    crossing_edges = 0
    crossing_weight = 0
    for nb in Gc.neighbors(node):
        cid = partition[nb]
        if cid != own:
            other_comms.add(cid)
            crossing_edges += 1
            crossing_weight += Gc[node][nb].get(WEIGHT_ATTR, 1)
    if not other_comms:
        continue
    wdeg = weighted_degree.get(node, 0)
    bridge_rows.append({
        'stop_id': node,
        'stop_name': G.nodes[node]['stop_name'],
        'own_community': own,
        'own_community_label': label_of.get(own, ''),
        'connected_communities': len(other_comms),
        'crossing_edges': crossing_edges,
        'crossing_weight': int(crossing_weight),
        'degree': Gc.degree(node),
        'weighted_degree': int(wdeg),
        'crossing_weight_share': round(crossing_weight / wdeg, 4) if wdeg else None,
        'lat': G.nodes[node]['lat'],
        'lon': G.nodes[node]['lon'],
        'region': G.nodes[node]['region'],
        'metro': G.nodes[node]['metro'],
    })

bridges_df = (pd.DataFrame(bridge_rows)
              .sort_values(['connected_communities', 'crossing_weight'], ascending=False)
              .reset_index(drop=True))
bridges_df.to_csv(TABLES / 'inter_community_bridges.csv', index=False, encoding='utf-8-sig')

cross_edges = sum(1 for u, v in Gc.edges() if partition[u] != partition[v])
cross_weight = sum(d.get(WEIGHT_ATTR, 1) for u, v, d in Gc.edges(data=True)
                   if partition[u] != partition[v])
component_weight = sum(d.get(WEIGHT_ATTR, 1) for _, _, d in Gc.edges(data=True))

print(f"saved {TABLES / 'inter_community_bridges.csv'}  ({len(bridges_df):,} stations)")
print(f'boundary stations     : {len(bridges_df):,} of {Gc.number_of_nodes():,} '
      f'({len(bridges_df) / Gc.number_of_nodes():.2%} of the analysed component)')
print(f'cross-community edges : {cross_edges:,} of {Gc.number_of_edges():,} '
      f'({cross_edges / Gc.number_of_edges():.2%})')
print(f'cross-community trips : {cross_weight:,} of {component_weight:,} '
      f'({cross_weight / component_weight:.2%} of all trip weight)')
bridges_df.head(TOP_N)[['stop_id', 'stop_name', 'own_community_label', 'connected_communities',
                        'crossing_edges', 'crossing_weight', 'degree']]


## 13. The community map

Every station in the analysed component drawn at its true longitude and latitude, coloured by community; stations outside the component are drawn in pale grey so a reader can see what was excluded. This is the figure that decides whether the partition means anything: if Louvain has found real structure, the colours must form **spatially contiguous blobs**, because a transport network's communities are geographic by construction - buses connect places that are near each other. Scattered confetti would mean the partition is numerical noise.

Two deliberate changes from the original script. First, it built its colour map with `tab20.resampled(n)`, which *interpolates* between the twenty categorical colours and produces dozens of near-identical muddy shades once there are more communities than colours. We instead cycle through the three 20-colour qualitative maps (60 visually distinct colours) modulo the community id: colours therefore repeat, but because ids are ordered by size, the large communities that dominate the picture all get different colours. The map is meant to show *contiguity*, not to let a reader identify a specific community by its colour - the annotated labels do that job. Second, the original filtered coordinates with `if lat and lon`, which silently drops a stop at exactly `0.0` and, worse, keeps `NaN`; we test explicitly for a present, finite value.

The largest few communities are annotated in place, at their centroids, with their data-derived labels.


In [ ]:
# --- figures/community_map_louvain.png -------------------------------------
def has_coords(node):
    """True only when both coordinates are present and finite (0.0 included)."""
    lat, lon = G.nodes[node]['lat'], G.nodes[node]['lon']
    return (lat is not None and lon is not None
            and np.isfinite(lat) and np.isfinite(lon))


qualitative = [c for name in ('tab20', 'tab20b', 'tab20c')
               for c in matplotlib.colormaps[name].colors]

plot_nodes = [n for n in Gc.nodes() if has_coords(n)]
if not plot_nodes:
    raise ValueError('No station in the analysed component carries usable coordinates - '
                     'check nodes.csv from notebook 02.')
plot_lons = [G.nodes[n]['lon'] for n in plot_nodes]
plot_lats = [G.nodes[n]['lat'] for n in plot_nodes]
plot_cols = [qualitative[partition[n] % len(qualitative)] for n in plot_nodes]
bg_nodes = [n for n in G.nodes() if n not in partition and has_coords(n)]

fig, ax = plt.subplots(figsize=(8.5, 11))
if bg_nodes:
    ax.scatter([G.nodes[n]['lon'] for n in bg_nodes], [G.nodes[n]['lat'] for n in bg_nodes],
               s=MAP_POINT_SIZE, color='#cbd5e1', alpha=0.6, linewidths=0,
               label=f'outside largest component (n={len(bg_nodes):,})')
ax.scatter(plot_lons, plot_lats, s=MAP_POINT_SIZE, color=plot_cols,
           alpha=MAP_ALPHA, linewidths=0)

for _, row in summary_df.head(ANNOTATE_TOP).iterrows():
    if row['centroid_lat'] is None or row['centroid_lon'] is None:
        continue
    ax.annotate(f"{row['label']}\n(n={int(row['size']):,})",
                (row['centroid_lon'], row['centroid_lat']),
                fontsize=8, fontweight='bold', ha='center', zorder=6,
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.75,
                          edgecolor='#94a3b8', linewidth=0.5))

ax.set_aspect(1 / np.cos(np.deg2rad(float(np.mean(plot_lats)))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Louvain communities of the Israeli public-transport network\n'
             f'{num_communities:,} communities, modularity Q = {modularity:.4f} '
             f'(seed {LOUVAIN_SEED}, resolution {LOUVAIN_RESOLUTION})')
if bg_nodes:
    ax.legend(loc='upper left', fontsize=8, markerscale=3)
plt.tight_layout()
plt.savefig(FIGURES / 'community_map_louvain.png', dpi=FIG_DPI)
plt.show()

print(f'plotted {len(plot_nodes):,} of {Gc.number_of_nodes():,} analysed stations '
      f'({Gc.number_of_nodes() - len(plot_nodes):,} lack usable coordinates)')


## 14. The community size distribution

Two views of how the stations are split up. The **left panel** ranks the largest communities by size, labelled with the data-derived label and the busiest station of each, so a reader can see immediately whether the partition is dominated by a few metropolitan blocks. The **right panel** shows the whole rank-size distribution on logarithmic axes - every community, not just the top ones - which is the honest view: it reveals whether there is a long tail of tiny communities and roughly how fast size falls with rank.

The tail matters for interpreting section 8. Communities of a handful of stations are exactly the ones that merge or split when the seed changes, so a fat tail of tiny communities is a direct visual explanation of why the total count is unstable while the big blocks are not. The horizontal reference line marks a size of ten stations, the threshold reported in the headline printout of section 7.


In [ ]:
# --- figures/community_size_distribution.png -------------------------------
top = summary_df.head(TOP_N).iloc[::-1]
top_labels = [f'#{int(cid)} {lab} - {hub}' for cid, lab, hub
              in zip(top['community_id'], top['label'], top['hub_stop_name'])]

sizes_sorted = summary_df['size'].to_numpy()
ranks = np.arange(1, len(sizes_sorted) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 8))

bars = axes[0].barh(range(len(top)), top['size'].to_numpy(), color='#7c3aed')
axes[0].set_yticks(range(len(top)))
axes[0].set_yticklabels(top_labels, fontsize=8)
for bar, val in zip(bars, top['size'].to_numpy()):
    axes[0].text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                 f' {int(val):,}', va='center', fontsize=8)
axes[0].set_xlabel('Number of stations')
axes[0].set_title(f'Largest {len(top)} communities of {num_communities:,}')
axes[0].margins(x=0.14)

axes[1].scatter(ranks, sizes_sorted, s=18, color='#2563eb', alpha=0.75)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].axhline(10, color='#dc2626', linestyle='--', linewidth=1, label='size = 10 stations')
axes[1].set_xlabel('Community rank (log scale)')
axes[1].set_ylabel('Number of stations (log scale)')
axes[1].set_title('Rank-size distribution of all communities')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'community_size_distribution.png', dpi=FIG_DPI)
plt.show()

print(summary_df['size'].describe().to_string())


## 15. Where the boundary stations are

The final figure places the highest-ranked inter-community bridge stations on the map, sized and coloured by how many communities they touch, over a pale background of the whole network. Since communities in a transport network are geographic, these points should fall on the *seams* between the coloured blobs of section 13 - the corridors linking one metropolitan cluster to the next. Those seams are the places where a failure does not merely delay a journey but forces it out of its cluster entirely, which is why this table feeds the resilience notebooks.


In [ ]:
# --- figures/inter_community_bridges_map.png -------------------------------
top_bridges = bridges_df.dropna(subset=['lat', 'lon']).head(100)

fig, ax = plt.subplots(figsize=(8.5, 11))
bg = [n for n in G.nodes() if has_coords(n)]
ax.scatter([G.nodes[n]['lon'] for n in bg], [G.nodes[n]['lat'] for n in bg],
           s=1, color='#e2e8f0', alpha=0.5, linewidths=0)
sc = ax.scatter(top_bridges['lon'], top_bridges['lat'],
                s=top_bridges['connected_communities'] * 20,
                c=top_bridges['connected_communities'], cmap='Reds',
                edgecolors='black', linewidths=0.3, zorder=5)
plt.colorbar(sc, ax=ax, label='distinct communities touched')
ax.set_aspect(1 / np.cos(np.deg2rad(float(np.mean(plot_lats)))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Top {len(top_bridges)} inter-community bridge stations\n'
             '(marker size = number of communities the station connects)')
plt.tight_layout()
plt.savefig(FIGURES / 'inter_community_bridges_map.png', dpi=FIG_DPI)
plt.show()


## 16. Headline statistics

Everything worth quoting from this stage, collected into one dictionary and written both as JSON and as a one-row CSV. Every field is computed above - and the seed, the resolution and the backend are stored *alongside* the results precisely so that a future reader can tell which configuration produced them, which is the discipline whose absence caused the 91-versus-73 confusion in the first place.


In [ ]:
# --- community_detection_summary.json --------------------------------------
summary = {
    'louvain_backend': LOUVAIN_BACKEND,
    'louvain_seed': LOUVAIN_SEED,
    'louvain_resolution': LOUVAIN_RESOLUTION,
    'weighted': True,
    'graph_nodes': G.number_of_nodes(),
    'graph_edges': G.number_of_edges(),
    'connected_components': len(components),
    'analysed_nodes': Gc.number_of_nodes(),
    'analysed_edges': Gc.number_of_edges(),
    'num_communities': num_communities,
    'modularity': round(modularity, 4),
    'largest_community_size': int(community_sizes.iloc[0]),
    'largest_community_share': round(float(community_sizes.iloc[0]) / Gc.number_of_nodes(), 4),
    'median_community_size': float(community_sizes.median()),
    'communities_under_10_stations': int((community_sizes < 10).sum()),
    'cross_community_edges': int(cross_edges),
    'cross_community_edge_share': round(cross_edges / Gc.number_of_edges(), 4),
    'cross_community_weight_share': round(cross_weight / component_weight, 4),
    'inter_community_stations': int(len(bridges_df)),
    'inter_community_station_share': round(len(bridges_df) / Gc.number_of_nodes(), 4),
    'max_communities_touched_by_one_station':
        int(bridges_df['connected_communities'].max()) if len(bridges_df) else 0,
    'seed_sweep_seeds': list(SEED_SWEEP),
    'seed_sweep_min_communities': int(seed_runs['num_communities'].min()),
    'seed_sweep_max_communities': int(seed_runs['num_communities'].max()),
    'seed_sweep_min_modularity': float(seed_runs['modularity'].min()),
    'seed_sweep_max_modularity': float(seed_runs['modularity'].max()),
    'seed_sweep_min_ari_vs_headline': float(seed_runs['ari_vs_headline'].min()),
    'label_propagation_communities': int(len(set(partition_lp.values()))) if partition_lp else None,
    'label_propagation_modularity': round(modularity_lp, 4) if modularity_lp is not None else None,
    'label_propagation_ari_vs_louvain': round(ari_lp, 4) if ari_lp is not None else None,
}

with open(STAGE / 'community_detection_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
flat = {k: (json.dumps(v) if isinstance(v, list) else v) for k, v in summary.items()}
pd.DataFrame([flat]).to_csv(TABLES / 'community_detection_summary.csv',
                            index=False, encoding='utf-8-sig')

print('saved:', STAGE / 'community_detection_summary.json')
for p in sorted(TABLES.iterdir()) + sorted(FIGURES.iterdir()):
    print(f'  {p.relative_to(STAGE)}  ({p.stat().st_size / 1024:,.0f} KB)')
pd.DataFrame({'metric': list(summary.keys()), 'value': [str(v) for v in summary.values()]})


## Takeaways

* **The network does decompose into communities, and the modularity value is the claim worth making.** The headline run prints its modularity `Q` in section 7; read it against the usual benchmark that values roughly in the 0.3-0.7 range indicate genuine community structure, while a value near 0 would mean the partition is no better than chance. Modularity, not the community count, is the reproducible quantity - it barely moved across the seed sweep.

* **The number of communities is not a property of the network.** This is the direct answer to the 91-versus-73 disagreement between the Hebrew final report and the presentation pipeline. Louvain is a stochastic greedy heuristic over a non-convex objective, so the seed alone shifts the count while the modularity stays essentially flat and the adjusted Rand index against the headline run stays high: the partition is stable, only its bookkeeping at the margins is not. On top of that, the resolution `gamma` moves the count deliberately and by much more; running on the full graph rather than the largest connected component adds one community per isolated component (the exact offset is printed in section 6); and `python-louvain`, `networkx` and igraph/Leiden reach different local optima from identical input. Either earlier number is reproducible under some combination of those settings, which is why this notebook pins the seed and the resolution, records the backend, and reports its own result rather than quoting a remembered figure. **The correct way to cite a Louvain result is "N communities at seed S, resolution R, implementation X, modularity Q", never "N communities".**

* **Communities are geographic, which is the sanity check that matters.** The map colours form contiguous spatial blobs rather than confetti, and the derived labels - each community's modal metropolitan area and modal region, computed at run time - line up with the metropolitan structure of the country. That is expected for a transport network, where adjacency is constrained by physical distance, and it is the main evidence that the partition is meaningful rather than numerical noise. The caveat is that those labels inherit notebook 01's crude latitude/longitude region rule and its four metropolitan discs, so a low `dominant_metro_share` in `community_summary.csv` marks a community whose one-word label is genuinely a simplification.

* **The size distribution is very uneven, and the tail explains the instability.** A few large metropolitan communities hold most of the stations while a long tail of small ones covers the periphery, as the rank-size panel shows. Those small communities are exactly the ones that merge or split between seeds, so the fat tail and the unstable count are the same phenomenon seen twice. It also means summary statistics over communities - a mean community size, for instance - are close to meaningless here; the distribution has to be shown, not averaged.

* **Boundary stations are a minority, and that is the resilience finding.** Only a fraction of stations have any neighbour in another community, and only a fraction of the trip weight crosses a community boundary; the exact shares are printed in section 12 and stored in the summary JSON. A network whose clusters are joined by relatively few stations is efficient but exposed: those stations carry the inter-regional traffic, and the ones touching the most communities are natural first candidates for the targeted-attack experiments in the robustness notebooks. Note that these are *not* the same objects as notebook 03's bridges - those were edges whose removal disconnects the graph, these are stations that carry cross-cluster flow - and joining the two tables identifies the stations that are both.

* **Honest limits.** The partition is derived from *scheduled trip counts on a single GTFS snapshot*, with no passenger loads, no timetable, no transfer times and no walking links between nearby stops - two stops on opposite sides of the same junction are separate nodes unless a trip connects them, which fragments the graph in a way real passengers do not experience. Louvain also suffers from the well-known resolution limit: at `gamma = 1` it cannot resolve communities much smaller than the square root of the total edge weight, so genuinely small local clusters may be absorbed into larger neighbours. Label propagation, which optimises nothing and ignores weights, is reported alongside as an independent check rather than as a competitor: where the two agree, the structure is safe to talk about; where they disagree, the granularity of any single method should not be over-interpreted.
